In [1]:
import os
import math
import xml.etree.ElementTree as ET
from pycram.datastructures.enums import GripperType, WorldMode
from jinja2 import Template
import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display
from ipywidgets import Dropdown, VBox, Label, HBox
import numpy as np
from scipy.spatial.transform import Rotation as R
import subprocess

from pycram.ros.viz_marker_publisher import VizMarkerPublisher
from pycram.ros.tf_broadcaster import TFBroadcaster
from pycram.worlds.bullet_world import BulletWorld
from pycram.process_module import simulated_robot
from pycram.datastructures.pose import Pose
from pycram.object_descriptors.urdf import ObjectDescription
from pycram.world_concepts.world_object import Object
from pycram.datastructures.enums import ObjectType


class RobotURDF:
    def __init__(self, urdf_path):
        self.joint_limits = {}
        self.joint_types = {}
        self.urdf_path = urdf_path
        self.root_link = None
        self.end_links = []
        self.links = []
        self.joints = {}
        self.link_parents = {}
        self.link_children = {}
        self.joint_parents = {}
        self.joint_children = {}
        self.parse_urdf()

    def parse_urdf(self):
        tree = ET.parse(self.urdf_path)
        root = tree.getroot()

        for link in root.findall('.//link'):
            link_name = link.get('name')
            self.links.append(link_name)

        child_links = set()
        parent_links = set()

        for joint in root.findall('.//joint'):
            joint_name = joint.get('name')
            self.joints[joint_name] = joint
            parent_link = joint.find('parent').get('link')
            child_link = joint.find('child').get('link')
            parent_links.add(parent_link)
            child_links.add(child_link)
            self.joint_parents[joint_name] = parent_link
            self.joint_children[joint_name] = child_link
            self.link_parents[child_link] = parent_link
            self.link_children.setdefault(parent_link, []).append(child_link)

            joint_type = joint.get('type')
            if joint_type != 'fixed':
                self.joint_types[joint_name] = joint_type
                if joint_type == 'continuous':
                    self.joint_limits[joint_name] = (-math.pi, math.pi)
                else:
                    limit = joint.find('limit')
                    if limit is not None:
                        lower = float(limit.get('lower', '-math.inf'))
                        upper = float(limit.get('upper', 'math.inf'))
                        self.joint_limits[joint_name] = (lower, upper)

        root_links = set(self.links) - child_links
        self.root_link = root_links.pop() if root_links else None
        self.end_links = list(set(self.links) - parent_links)

    def get_kinematic_chain_to_link(self, target_link):
        path = []
        found = False

        def dfs(current_link):
            nonlocal found
            if current_link == target_link:
                found = True
                return True
            for child in self.link_children.get(current_link, []):
                joint_name = next((j for j, p in self.joint_parents.items()
                                   if p == current_link and self.joint_children[j] == child), None)
                if joint_name:
                    path.append((joint_name, child))
                    if dfs(child):
                        return True
                    path.pop()
            return False

        dfs(self.root_link)
        return path if found else None

    def get_joints_between_links(self, start_link, end_link):
        path = []
        found = False

        def dfs(current_link):
            nonlocal found
            if current_link == end_link:
                found = True
                return True
            for child in self.link_children.get(current_link, []):
                joint_name = next((j for j, p in self.joint_parents.items()
                                   if p == current_link and self.joint_children[j] == child), None)
                if joint_name:
                    if self.joint_types.get(joint_name) != 'fixed':
                        path.append(joint_name)
                    if dfs(child):
                        return True
                    if self.joint_types.get(joint_name) != 'fixed':
                        path.pop()
            return False

        dfs(start_link)
        return path if found else None

    def get_descendant_joints(self, link_name, exclude_links=None):
        if exclude_links is None:
            exclude_links = set()
        else:
            exclude_links = set(exclude_links)

        joints = []

        def dfs(current_link):
            for child in self.link_children.get(current_link, []):
                if child in exclude_links:
                    continue
                joint_name = next((j for j, p in self.joint_parents.items()
                                   if p == current_link and self.joint_children[j] == child), None)
                if joint_name:
                    if self.joint_types.get(joint_name) != 'fixed':
                        joints.append(joint_name)
                    dfs(child)

        dfs(link_name)
        return joints

def get_git_root(path="."):
    """
    Get the root directory of the Git repository.

    Args:
        path (str): Path to start looking for the Git repository. Defaults to the current directory.

    Returns:
        str: The root directory of the Git repository, or None if not inside a Git repository.
    """
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=path,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError:
        return None

def generate_robot_file(variables):
    """
    Generate a Python file using a Jinja2 template.

    Args:
        template_path (str): Path to the Jinja2 template file.
        output_path (str): Path to save the generated Python file.
        variables (dict): Dictionary of variables to render the template.

    Returns:
        None
    """
    robot_name = variables.get("robot_name")
    if not robot_name:
        print("Robot name not provided.")
        return
    
    root_dir = get_git_root()    
    output_path = root_dir + f"/src/pycram/robot_descriptions/{robot_name}_description.py"
    template_path = root_dir + "/src/pycram/robot_descriptions/template/robot_description_template.py.jinja"

    with open(template_path, 'r') as file:
        template_content = file.read()

    template = Template(template_content)

    rendered_content = template.render(**variables)

    try:
        with open(output_path, 'w') as output_file:
            output_file.write(rendered_content)
        print(f"Generated file successfully at: {output_path}")
    except Exception as e:
        print(f"Failed to write file: {e}")

def round_to_nearest_90(x):
    return round(x / (math.pi / 2)) * (math.pi / 2)

def round_quaternion_to_nearest_90(quaternion):
    rotation = R.from_quat(quaternion)
    roll, pitch, yaw = rotation.as_euler('xyz')
    roll = round_to_nearest_90(roll)
    pitch = round_to_nearest_90(pitch)
    yaw = round_to_nearest_90(yaw)
    return R.from_euler('xyz', [roll, pitch, yaw]).as_quat()

def find_aligned_axis(rounded_quaternion):
    rotation_matrix = R.from_quat(rounded_quaternion).as_matrix()    
    return rotation_matrix[0]


Failed to import Robokudo messages, the real robot will not be available
pybullet build time: Nov 28 2023 23:51:11


In [2]:
from pycram.ros.viz_marker_publisher import AxisMarkerPublisher


class RobotDescriptionBuilder:
    def __init__(self, urdf_path):
        self.robot_urdf = RobotURDF(urdf_path)
        self.variables = {
            "robot_name": "",
            "root_frame": "",
            "torso_start_link": "",
            "torso_start_joint": "",
            "start_torso_parent_link": "",
            "torso_end_link": "",
            "left_arm_end_link": "",
            "right_arm_end_link": "",
            "left_gripper_parent_link": "",
            "right_gripper_parent_link": "",
            "left_tool_frame": "",
            "right_tool_frame": "",
            "left_gripper_open_states": {},
            "left_gripper_close_states": {},
            "right_gripper_open_states": {},
            "right_gripper_close_states": {},
            "left_arm_park_states": {},
            "right_arm_park_states": {},
            "left_gripper_type": GripperType.PARALLEL,
            "right_gripper_type": GripperType.PARALLEL,
            "left_gripper_opening_distance": 0.0,
            "right_gripper_opening_distance": 0.0,
            "torso_high_states": {},
            "torso_mid_states": {},
            "torso_low_states": {},
            "camera_name": "",
            "camera_link": "",
            "min_height": 0.0,
            "max_height": 0.0,
            "horizontal_angle": 0.0,
            "vertical_angle": 0.0,
            "front_facing_axis": [0, 0, 0],
            "lower_neck_link": "",
            "upper_neck_link": "",
            "neck_yaw_joint": "",
            "neck_pitch_joint": "",
            "right_grasp_orientation": [0, 0, 0, 0],
            "left_grasp_orientation": [0, 0, 0, 0],
            "costmap_offset": 0.0,
            "left_palm_axis": [0, 0, 0],
            "right_palm_axis": [0, 0, 0]
        }

        self.simulation_initialized = False
        self.initialize_simulation()

    def initialize_simulation(self):
        extension = ObjectDescription.get_file_extension()
        self.world = BulletWorld(WorldMode.DIRECT)
        self.viz = VizMarkerPublisher()
        self.tf = TFBroadcaster()

        robot_name = os.path.splitext(os.path.basename(self.robot_urdf.urdf_path))[0]
        self.variables["robot_name"] = robot_name 
        self.robot = Object(robot_name, ObjectType.ROBOT, f"{robot_name}{extension}", pose=Pose([0, 0, 0]))
        self.initial_joint_states = {
            joint_name: 0.0 for joint_name in self.robot_urdf.joint_types.keys()
        }
        self.simulation_initialized = True
        
    def debug_variables(self):
        print("Current Variables State:")
        for key, value in self.variables.items():
            print(f"{key}: {value}")

    
    def setup_torso_gui(self):
        """Set up the GUI for torso-related functionality with improved styling and output layout."""
        dropdown_layout = Layout(
            width="95%", 
            background="#dff2f8", 
            border="1px solid #003152"
        )
        
        slider_layout = Layout(
            width="95%", 
            background="#dff2f8", 
            border="1px solid #003152"
        )
        
        label_style = {
            "description_width": "150px",
            "color": "#003152",
            "background": "#dff2f8"
        }
        
        output_layout = Layout(
            width="95%",
            border="1px solid #003152",
            padding="10px",
            background="#dff2f8",
            color="#003152"
        )
        robot_name_label = widgets.Label(f"Robot Name: {self.variables['robot_name']}")
    
        root_frame_options = self.robot_urdf.links
        default_root_frame = self.robot_urdf.root_link
        root_frame_dropdown = Dropdown(
            options=root_frame_options,
            value=default_root_frame,
            description="Root Frame:",
            layout=dropdown_layout,
            style=label_style
        )
    
        def on_root_frame_change(change):
            self.variables["root_frame"] = change.new
            update_torso_links(change.new)
    
        root_frame_dropdown.observe(on_root_frame_change, names='value')
        self.variables["root_frame"] = default_root_frame
    
        torso_start_link_dropdown = Dropdown(
            options=[],
            value=None,
            description="Torso Start Link:",
            layout=dropdown_layout,
            style=label_style
        )
        torso_end_link_dropdown = Dropdown(
            options=[],
            value=None,
            description="Torso End Link:",
            layout=dropdown_layout,
            style=label_style
        )
    
        def update_torso_links(root_link):
            descending_links = [
                self.robot_urdf.joint_children[joint]
                for joint in self.robot_urdf.get_descendant_joints(root_link)
            ]
            torso_start_link_dropdown.options = descending_links
            torso_end_link_dropdown.options = descending_links
            if descending_links:
                torso_start_link_dropdown.value = descending_links[0]
                torso_end_link_dropdown.value = descending_links[0]
    
        kinematic_chain_output = widgets.Output(layout=output_layout)
        sliders_output_high = widgets.Output(layout=output_layout)
        sliders_output_mid = widgets.Output(layout=output_layout)
        sliders_output_low = widgets.Output(layout=output_layout)
    
        def update_torso_chain(*args):
            kinematic_chain_output.clear_output()
            sliders_output_high.clear_output()
            sliders_output_mid.clear_output()
            sliders_output_low.clear_output()
    
            start_link = torso_start_link_dropdown.value
            end_link = torso_end_link_dropdown.value
    
            if start_link:
                self.variables["torso_high_states"] = {}
                self.variables["torso_mid_states"] = {}
                self.variables["torso_low_states"] = {}
                if start_link == end_link:
                    joint_name = next(
                        (joint for joint, parent in self.robot_urdf.joint_parents.items()
                         if parent == start_link or self.robot_urdf.joint_children[joint] == start_link),
                        None
                    )
                    parent_link = self.robot_urdf.link_parents.get(start_link, None)
    
                    self.variables["torso_start_link"] = start_link
                    self.variables["torso_end_link"] = end_link
                    self.variables["torso_start_joint"] = joint_name
                    self.variables["torso_end_joint"] = joint_name
                    self.variables["start_torso_parent_link"] = parent_link
    
                    with kinematic_chain_output:
                        if joint_name:
                            print(f"Single joint found: {joint_name} for link {start_link}")
                            print(f"Parent link: {parent_link}")
                    if joint_name:
                        add_sliders([joint_name])
                else:
                    joints = self.robot_urdf.get_joints_between_links(start_link, end_link)
                    if not joints:
                        torso_end_link_dropdown.value = start_link
                        self.variables["torso_end_link"] = start_link
                    else:
                        chain = self.robot_urdf.get_kinematic_chain_to_link(end_link)
                        parent_link = self.robot_urdf.link_parents.get(start_link, None)
    
                        self.variables["torso_start_link"] = start_link
                        self.variables["torso_end_link"] = end_link
                        self.variables["torso_start_joint"] = joints[0]
                        self.variables["torso_end_joint"] = joints[-1]
                        self.variables["start_torso_parent_link"] = parent_link
    
                        with kinematic_chain_output:
                            if joints:
                                print("Kinematic Chain:")
                                print(" -> ".join([f"{joint} ({link})" for joint, link in chain]))
                                print(f"Parent link of start torso: {parent_link}")
                        if joints:
                            add_sliders(joints)
    
        def add_sliders(joints):
            sliders_high, sliders_mid, sliders_low = [], [], []
    
            def create_slider(joint_name, state_key, default_value):
                joint_limits = self.robot_urdf.joint_limits.get(joint_name, (-math.pi, math.pi))
                slider = widgets.FloatSlider(
                    value=default_value,
                    min=joint_limits[0],
                    max=joint_limits[1],
                    step=0.01,
                    description=joint_name,
                    layout=slider_layout,
                    style=label_style
                )
    
                def on_slider_change(change):
                    joint_state = change.new
                    self.variables[state_key][joint_name] = joint_state
                    self.robot.set_joint_position(joint_name, joint_state)
    
                slider.observe(on_slider_change, names='value')
                self.variables[state_key][joint_name] = default_value
                self.robot.set_joint_position(joint_name, default_value)
                return slider
    
            for joint_name in joints:
                joint_limits = self.robot_urdf.joint_limits.get(joint_name, (-math.pi, math.pi))
                high_default = joint_limits[1]
                mid_default = (joint_limits[0] + joint_limits[1]) / 2
                low_default = joint_limits[0]
    
                sliders_high.append(create_slider(joint_name, "torso_high_states", high_default))
                sliders_mid.append(create_slider(joint_name, "torso_mid_states", mid_default))
                sliders_low.append(create_slider(joint_name, "torso_low_states", low_default))
    
            with sliders_output_high:
                print("High States:")
                for slider in sliders_high:
                    display(slider)
    
            with sliders_output_mid:
                print("Mid States:")
                for slider in sliders_mid:
                    display(slider)
    
            with sliders_output_low:
                print("Low States:")
                for slider in sliders_low:
                    display(slider)
    
        torso_start_link_dropdown.observe(update_torso_chain, names='value')
        torso_end_link_dropdown.observe(update_torso_chain, names='value')
    
        display(
            VBox([
                robot_name_label,
                root_frame_dropdown,
                widgets.HTML("<b>Torso Configuration</b>"),
                VBox([torso_start_link_dropdown, torso_end_link_dropdown], layout=output_layout),
                widgets.HTML("<b>Torso Kinematic Chain</b>"),
                kinematic_chain_output,
                widgets.HTML("<b>Torso High States</b>"),
                sliders_output_high,
                widgets.HTML("<b>Torso Mid States</b>"),
                sliders_output_mid,
                widgets.HTML("<b>Torso Low States</b>"),
                sliders_output_low
            ])
        )
    
        update_torso_links(default_root_frame)


    def setup_arm_gui(self):
        required_vars = ["torso_start_link", "torso_end_link"]
        for var in required_vars:
            if not self.variables.get(var):
                raise ValueError(f"Variable '{var}' is not set. Please complete torso setup first.")
    
        dropdown_layout = Layout(width="95%")
        label_style = {"description_width": "150px"}
        column_layout = Layout(width="50%")
        output_layout = Layout(width="95%", border="1px solid lightgray", padding="10px")
    
        self.variables["left_arm_identified"] = False
        self.variables["right_arm_identified"] = False
    
        def create_arm_setup(side):
            tool_frame_var = f"{side}_tool_frame"
            gripper_parent_var = f"{side}_gripper_parent_link"
            arm_end_var = f"{side}_arm_end_link"
            arm_identified_var = f"{side}_arm_identified"
    
            tool_frame_output = widgets.Output(layout=output_layout)
            gripper_parent_output = widgets.Output(layout=output_layout)
            arm_end_output = widgets.Output(layout=output_layout)
            kinematic_chain_output = widgets.Output(layout=output_layout)
            
            torso_start_link = self.variables.get("torso_start_link")
            descending_links = [
                self.robot_urdf.joint_children[joint]
                for joint in self.robot_urdf.get_descendant_joints(torso_start_link)
            ]
            
            toolframe_options = descending_links
            tool_frame_dropdown = Dropdown(
                options=[None] + toolframe_options,
                value=None,
                description=f"{side.capitalize()} Tool Frame:",
                layout=dropdown_layout,
                style=label_style
            )
    
            def on_tool_frame_change(change):
                tool_frame = change.new
                self.variables[tool_frame_var] = tool_frame
                self.variables[arm_identified_var] = tool_frame is not None
                update_kinematic_chain()
    
            tool_frame_dropdown.observe(on_tool_frame_change, names='value')
    
            with tool_frame_output:
                display(tool_frame_dropdown)
    
            gripper_parent_dropdown = Dropdown(
                options=[],
                value=None,
                description=f"{side.capitalize()} Gripper Link:",
                layout=dropdown_layout,
                style=label_style
            )
    
            def on_gripper_parent_change(change):
                self.variables[gripper_parent_var] = change.new
                new_parent = change.new
                new_arm_end_parent = self.robot_urdf.link_parents.get(new_parent)
                if new_parent:
                    torso_start_link = self.variables.get("torso_start_link")
                    arm_chain = self.robot_urdf.get_joints_between_links(torso_start_link, new_parent)
                    arm_links = [self.robot_urdf.joint_children[joint] for joint in arm_chain] if arm_chain else []
                    arm_end_dropdown.options = [None] + arm_links
                    arm_end_dropdown.value = new_arm_end_parent
    
            gripper_parent_dropdown.observe(on_gripper_parent_change, names='value')
    
            with gripper_parent_output:
                display(gripper_parent_dropdown)
    
            arm_end_dropdown = Dropdown(
                options=[],
                value=None,
                description=f"{side.capitalize()} Arm End Link:",
                layout=dropdown_layout,
                style=label_style
            )
    
            def on_arm_end_change(change):
                self.variables[arm_end_var] = change.new
    
            arm_end_dropdown.observe(on_arm_end_change, names='value')
    
            with arm_end_output:
                display(arm_end_dropdown)
    
            def update_kinematic_chain(*args):
                kinematic_chain_output.clear_output()
                torso_start_link = self.variables.get("torso_start_link")
                tool_frame = tool_frame_dropdown.value
    
                if torso_start_link and tool_frame:
                    tool_frame_chain = self.robot_urdf.get_joints_between_links(torso_start_link, tool_frame)
                    tool_frame_links = [self.robot_urdf.joint_children[joint] for joint in tool_frame_chain] if tool_frame_chain else []
                    gripper_parent = tool_frame

                    if not gripper_parent:
                        with kinematic_chain_output:
                            print(f"Error: {side.capitalize()} tool frame has no parent link.")
                        return
    
                    gripper_chain = self.robot_urdf.get_joints_between_links(torso_start_link, gripper_parent)
                    gripper_links = [self.robot_urdf.joint_children[joint] for joint in gripper_chain] if gripper_chain else []
    
                    arm_end_parent = self.robot_urdf.link_parents.get(gripper_parent)
                    if not arm_end_parent:
                        with kinematic_chain_output:
                            print(f"Error: {side.capitalize()} gripper link has no parent link.")
                        return
    
                    arm_chain = self.robot_urdf.get_joints_between_links(torso_start_link, arm_end_parent)
                    arm_links = [self.robot_urdf.joint_children[joint] for joint in arm_chain] if arm_chain else []
    
                    gripper_parent_dropdown.options = [None] + gripper_links
                    gripper_parent_dropdown.value = gripper_parent
    
                    arm_end_dropdown.options = [None] + arm_links
                    arm_end_dropdown.value = arm_end_parent
    
                    self.variables[gripper_parent_var] = gripper_parent
                    self.variables[arm_end_var] = arm_end_parent
    
                    with kinematic_chain_output:
                        if tool_frame_chain:
                            print(f"{side.capitalize()} Kinematic Chain to Gripper Tool Frame:")
                            print(" -> ".join(tool_frame_links))
    
            return VBox([
                widgets.HTML(f"<b>{side.capitalize()} Arm Configuration</b>"),
                tool_frame_output,
                gripper_parent_output,
                arm_end_output,
                widgets.HTML(f"<b>{side.capitalize()} Kinematic Chain</b>"),
                kinematic_chain_output
            ], layout=column_layout)
    
        left_arm_setup = create_arm_setup("left")
        right_arm_setup = create_arm_setup("right")
    
        display(HBox([left_arm_setup, right_arm_setup]))


    
            
    def setup_arm_park_states(self):
        required_vars = ["torso_end_link"]
        for var in required_vars:
            if not self.variables.get(var):
                raise ValueError(f"Variable '{var}' is not set. Please complete torso setup first.")
    
        slider_layout = Layout(width="95%")
        label_style = {"description_width": "150px"}
        column_layout = Layout(width="50%")
        output_layout = Layout(width="95%", border="1px solid lightgray", padding="10px")
    
        def create_arm_park_setup(side):
            arm_identified_var = f"{side}_arm_identified"
            if not self.variables.get(arm_identified_var, False):
                return VBox([widgets.HTML(f"<b>{side.capitalize()} Arm Not Identified</b>")], layout=column_layout)
    
            torso_end_link = self.variables["torso_end_link"]
            arm_end_link = self.variables[f"{side}_arm_end_link"]
            park_states_var = f"{side}_arm_park_states"
    
            sliders_output = widgets.Output(layout=output_layout)
    
            kinematic_chain = self.robot_urdf.get_joints_between_links(torso_end_link, arm_end_link)
    
            kinematic_chain = [joint for joint in kinematic_chain if joint in self.robot_urdf.joint_types]
    
            if not kinematic_chain:
                sliders_output.clear_output()
                with sliders_output:
                    print(f"Error: No valid non-fixed joints found for {side.capitalize()} arm.")
                return VBox([widgets.HTML(f"<b>{side.capitalize()} Arm Park States</b>"), sliders_output], layout=column_layout)
    
            def create_slider(joint_name):
                joint_limits = self.robot_urdf.joint_limits.get(joint_name, (-math.pi, math.pi))
                if joint_limits[0] <= 0 <= joint_limits[1]:
                    default_value = 0
                else:
                    default_value = (joint_limits[0] + joint_limits[1]) / 2
                self.variables[park_states_var] = {}
    
                slider = widgets.FloatSlider(
                    value=default_value,
                    min=joint_limits[0],
                    max=joint_limits[1],
                    step=0.01,
                    description=joint_name,
                    layout=slider_layout,
                    style=label_style
                )
    
                def on_slider_change(change):
                    joint_state = change.new
                    self.variables[park_states_var][joint_name] = joint_state
                    self.robot.set_joint_position(joint_name, joint_state)
    
                slider.observe(on_slider_change, names='value')
                self.variables[park_states_var][joint_name] = default_value
                self.robot.set_joint_position(joint_name, default_value)
                return slider
    
            sliders = [create_slider(joint_name) for joint_name in kinematic_chain]
    
            sliders_output.clear_output()
            with sliders_output:
                for slider in sliders:
                    display(slider)
    
            return VBox([widgets.HTML(f"<b>{side.capitalize()} Arm Park States</b>"), sliders_output], layout=column_layout)
    
        left_arm_park_setup = create_arm_park_setup("left")
        right_arm_park_setup = create_arm_park_setup("right")
    
        display(HBox([left_arm_park_setup, right_arm_park_setup]))

                
    def setup_gripper_configuration(self):
        slider_layout = Layout(width="95%")
        label_style = {"description_width": "150px"}
        column_layout = Layout(width="50%")
        dropdown_layout = Layout(width="95%")
        output_layout = Layout(width="95%", border="1px solid lightgray", padding="10px")
    
        gripper_type_options = [
            GripperType.PARALLEL,
            GripperType.SUCTION,
            GripperType.FINGER,
            GripperType.HYDRAULIC,
            GripperType.PNEUMATIC,
            GripperType.CUSTOM
        ]
        
        for side in ["left", "right"]:
            open_states_var = f"{side}_gripper_open_states" 
            self.variables[open_states_var] = {}
            close_states_var = f"{side}_gripper_close_states"
            self.variables[close_states_var] = {}
    
        def create_gripper_setup(side):
            arm_identified_var = f"{side}_arm_identified"
            if not self.variables.get(arm_identified_var, False):
                return VBox([widgets.HTML(f"<b>{side.capitalize()} Gripper Not Identified</b>")], layout=column_layout)
            
            required_vars = [f"{side}_gripper_parent_link", f"{side}_tool_frame"]
            for var in required_vars:
                if not self.variables.get(var):
                    raise ValueError(f"Variable '{var}' is not set. Please complete arm and gripper setup first.")
    
            gripper_parent_link = self.variables[f"{side}_gripper_parent_link"]
            tool_frame = self.variables[f"{side}_tool_frame"]
            open_states_var = f"{side}_gripper_open_states"
            close_states_var = f"{side}_gripper_close_states"
            
            gripper_type_var = f"{side}_gripper_type"
    
            gripper_type_output = widgets.Output(layout=output_layout)
            open_states_output = widgets.Output(layout=output_layout)
            close_states_output = widgets.Output(layout=output_layout)
    
            gripper_type_dropdown = Dropdown(
                options=gripper_type_options,
                value=None,
                description=f"{side.capitalize()} Gripper Type:",
                layout=dropdown_layout,
                style=label_style
            )
    
            def on_gripper_type_change(change):
                self.variables[gripper_type_var] = change.new
    
            gripper_type_dropdown.observe(on_gripper_type_change, names='value')
    
            with gripper_type_output:
                display(gripper_type_dropdown)
    
            desc_joints = self.robot_urdf.get_descendant_joints(
                gripper_parent_link,
                exclude_links=[tool_frame]
            )
    
            relevant_joints = [joint for joint in desc_joints if joint in self.robot_urdf.joint_types]
    
            if not relevant_joints:
                return VBox([
                    gripper_type_output,
                    widgets.HTML(f"<b>No valid joints found for {side.capitalize()} Gripper</b>")
                ], layout=column_layout)
    
            def create_slider(joint_name, state_var, default_value):
                joint_limits = self.robot_urdf.joint_limits.get(joint_name, (-math.pi, math.pi))
    
                slider = widgets.FloatSlider(
                    value=default_value,
                    min=joint_limits[0],
                    max=joint_limits[1],
                    step=0.01,
                    description=joint_name,
                    layout=slider_layout,
                    style=label_style
                )
    
                def on_slider_change(change):
                    joint_state = change.new
                    self.variables[state_var][joint_name] = joint_state
                    self.robot.set_joint_position(joint_name, joint_state)
    
                slider.observe(on_slider_change, names='value')
                self.variables[state_var][joint_name] = default_value
                self.robot.set_joint_position(joint_name, default_value)
                return slider
    
            open_sliders = [create_slider(joint_name, open_states_var, self.robot_urdf.joint_limits[joint_name][1]) for joint_name in relevant_joints]
            close_sliders = [create_slider(joint_name, close_states_var, self.robot_urdf.joint_limits[joint_name][0]) for joint_name in relevant_joints]
    
            with open_states_output:
                widgets.HTML(f"<b>{side.capitalize()} Gripper Open States</b>")
                for slider in open_sliders:
                    display(slider)
    
            with close_states_output:
                widgets.HTML(f"<b>{side.capitalize()} Gripper Close States</b>")
                for slider in close_sliders:
                    display(slider)
    
            def swap_states(_):
                for joint in relevant_joints:
                    open_state = self.variables[open_states_var][joint]
                    close_state = self.variables[close_states_var][joint]
                    self.variables[open_states_var][joint] = close_state
                    self.variables[close_states_var][joint] = open_state
                    for slider in open_sliders:
                        if slider.description == joint:
                            slider.value = close_state
                    for slider in close_sliders:
                        if slider.description == joint:
                            slider.value = open_state
    
            swap_button = widgets.Button(
                description=f"Swap {side.capitalize()} Gripper Open/Close States",
                layout=Layout(width="95%")
            )
            swap_button.on_click(swap_states)

            return VBox([
                widgets.HTML(f"<b>{side.capitalize()} Gripper Open/Close States</b>"),
                gripper_type_output,
                open_states_output,
                widgets.VBox([swap_button], layout=output_layout),
                close_states_output
            ], layout=column_layout)
    
        left_gripper_setup = create_gripper_setup("left")
        right_gripper_setup = create_gripper_setup("right")
    
        display(HBox([left_gripper_setup, right_gripper_setup]))


    def setup_camera_configuration(self):
        required_vars = ["torso_high_states", "torso_low_states"]
        for var in required_vars:
            if not self.variables.get(var):
                raise ValueError(f"Variable '{var}' is not set. Please complete torso setup first.")
    
        dropdown_layout = Layout(width="95%")
        input_layout = Layout(width="95%")
        output_layout = Layout(width="95%", border="1px solid lightgray", padding="10px")
    
        leaf_links = sorted(self.robot_urdf.end_links)
        default_camera_link = leaf_links[0] if leaf_links else None
        self.variables["camera_link"] = default_camera_link
    
        camera_link_output = widgets.Output(layout=output_layout)
        with camera_link_output:
            camera_link_dropdown = Dropdown(
                options=[None] + leaf_links,
                value=default_camera_link,
                description="Camera Link:",
                layout=dropdown_layout,
                style={"description_width": "150px"}
            )
            display(camera_link_dropdown)
    
        self.variables["camera_name"] = default_camera_link or ""
        camera_name_output = widgets.Output(layout=output_layout)
        with camera_name_output:
            camera_name_input = widgets.Text(
                value=self.variables["camera_name"],
                description="Camera Name:",
                layout=input_layout,
                style={"description_width": "150px"}
            )
            display(camera_name_input)
    
        def on_camera_link_change(change):
            new_link = change.new
            self.variables["camera_link"] = new_link
            camera_name_input.value = new_link if new_link else ""
    
        def on_camera_name_change(change):
            self.variables["camera_name"] = change.new
    
        camera_link_dropdown.observe(on_camera_link_change, names='value')
        camera_name_input.observe(on_camera_name_change, names='value')
    
        min_height_output = widgets.Output(layout=output_layout)
        max_height_output = widgets.Output(layout=output_layout)
        calculate_heights_button_output = widgets.Output(layout=output_layout)
    
        self.variables["min_height"] = 0.0
        self.variables["max_height"] = 0.0
    
        with min_height_output:
            min_height_input = widgets.FloatText(
                value=0.0,
                description="Min Height:",
                layout=input_layout,
                style={"description_width": "150px"}
            )
            display(min_height_input)
    
        with max_height_output:
            max_height_input = widgets.FloatText(
                value=0.0,
                description="Max Height:",
                layout=input_layout,
                style={"description_width": "150px"}
            )
            display(max_height_input)
    
        def calculate_heights():
            camera_link = self.variables.get("camera_link")
            if not camera_link:
                return
    
            for joint, position in self.variables["torso_low_states"].items():
                self.robot.set_joint_position(joint, position)
            self.variables["min_height"] = self.robot.get_link_pose(link_name=camera_link).position_as_list()[2]
            min_height_input.value = self.variables["min_height"]
    
            for joint, position in self.variables["torso_high_states"].items():
                self.robot.set_joint_position(joint, position)
            self.variables["max_height"] = self.robot.get_link_pose(link_name=camera_link).position_as_list()[2]
            max_height_input.value = self.variables["max_height"]
    
        with calculate_heights_button_output:
            calculate_heights_button = widgets.Button(
                description="Calculate Heights",
                layout=Layout(width="95%")
            )
            calculate_heights_button.on_click(lambda _: calculate_heights())
            display(calculate_heights_button)
    
        horizontal_angle_output = widgets.Output(layout=output_layout)
        vertical_angle_output = widgets.Output(layout=output_layout)
    
        self.variables["horizontal_angle"] = 0.99483
        self.variables["vertical_angle"] = 0.75049
    
        with horizontal_angle_output:
            horizontal_angle_input = widgets.FloatText(
                value=0.99483,
                description="Horizontal Angle:",
                layout=input_layout,
                style={"description_width": "150px"}
            )
            display(horizontal_angle_input)
    
        with vertical_angle_output:
            vertical_angle_input = widgets.FloatText(
                value=0.75049,
                description="Vertical Angle:",
                layout=input_layout,
                style={"description_width": "150px"}
            )
            display(vertical_angle_input)
    
        def on_horizontal_angle_change(change):
            self.variables["horizontal_angle"] = change.new
    
        def on_vertical_angle_change(change):
            self.variables["vertical_angle"] = change.new
    
        horizontal_angle_input.observe(on_horizontal_angle_change, names='value')
        vertical_angle_input.observe(on_vertical_angle_change, names='value')
    
        front_facing_axis_output = widgets.Output(layout=output_layout)
        self.variables["front_facing_axis"] = [0, 0, 1]
    
        with front_facing_axis_output:
            front_facing_axis_dropdown = Dropdown(
                options=[
                    [1, 0, 0], [-1, 0, 0],
                    [0, 1, 0], [0, -1, 0],
                    [0, 0, 1], [0, 0, -1]
                ],
                value=[0, 0, 1],
                description="Front Facing Axis:",
                layout=dropdown_layout,
                style={"description_width": "150px"}
            )
            display(front_facing_axis_dropdown)
    
        def on_front_facing_axis_change(change):
            self.variables["front_facing_axis"] = change.new
    
        front_facing_axis_dropdown.observe(on_front_facing_axis_change, names='value')
    
        display(
            VBox([
                widgets.HTML(f"<b>Camera Setup</b>"),
                camera_link_output,
                camera_name_output,
                min_height_output,
                max_height_output,
                calculate_heights_button_output,
                horizontal_angle_output,
                vertical_angle_output,
                front_facing_axis_output
            ])
        )

        
    def setup_neck_configuration(self):
        if not self.variables.get("torso_end_link"):
            raise ValueError("Variable 'torso_end_link' is not set. Please complete torso setup first.")
    
        dropdown_layout = Layout(width="95%")
        output_layout = Layout(width="95%", border="1px solid lightgray", padding="10px")
        dropdown_style = {"description_width": "150px"}
    
        torso_end_link = self.variables["torso_end_link"]
        descending_links = self.robot_urdf.get_descendant_joints(torso_end_link, exclude_links=[])
        descending_links = [self.robot_urdf.joint_children[joint] for joint in descending_links]
    
        lower_neck_output = widgets.Output(layout=output_layout)
        with lower_neck_output:
            lower_neck_link_dropdown = Dropdown(
                options=[torso_end_link] + descending_links,
                value=torso_end_link,
                description="Lower Neck Parent Link:",
                layout=dropdown_layout,
                style=dropdown_style
            )
            display(lower_neck_link_dropdown)
    
        self.variables["lower_neck_link"] = torso_end_link
    
        def on_lower_neck_link_change(change):
            self.variables["lower_neck_link"] = change.new
            update_upper_neck_links(change.new)
    
        lower_neck_link_dropdown.observe(on_lower_neck_link_change, names='value')
    
        upper_neck_output = widgets.Output(layout=output_layout)
        self.variables["upper_neck_link"] = ""
        with upper_neck_output:
            upper_neck_link_dropdown = Dropdown(
                options=[""],
                value="",
                description="Upper Neck Link:",
                layout=dropdown_layout,
                style=dropdown_style
            )
            display(upper_neck_link_dropdown)
    
        def update_upper_neck_links(lower_link):
            descending_links_from_lower = self.robot_urdf.get_descendant_joints(lower_link, exclude_links=[])
            descending_links_from_lower = [self.robot_urdf.joint_children[joint] for joint in descending_links_from_lower]
            upper_neck_link_dropdown.options = [""] + descending_links_from_lower
            if descending_links_from_lower:
                upper_neck_link_dropdown.value = (
                    descending_links_from_lower[1] 
                    if len(descending_links_from_lower) > 1 
                    else descending_links_from_lower[0]
                )

    
        def on_upper_neck_link_change(change):
            self.variables["upper_neck_link"] = change.new
            update_kinematic_chain()
    
        upper_neck_link_dropdown.observe(on_upper_neck_link_change, names='value')
    
        neck_yaw_output = widgets.Output(layout=output_layout)
        neck_pitch_output = widgets.Output(layout=output_layout)
    
        self.variables["neck_yaw_joint"] = None
        self.variables["neck_pitch_joint"] = None
    
        with neck_yaw_output:
            neck_yaw_joint_dropdown = Dropdown(
                options=[],
                value=None,
                description="Neck Yaw Joint:",
                layout=dropdown_layout,
                style=dropdown_style
            )
            display(neck_yaw_joint_dropdown)
    
        with neck_pitch_output:
            neck_pitch_joint_dropdown = Dropdown(
                options=[],
                value=None,
                description="Neck Pitch Joint:",
                layout=dropdown_layout,
                style=dropdown_style
            )
            display(neck_pitch_joint_dropdown)
    
        def update_kinematic_chain():
            lower_neck_link = self.variables["lower_neck_link"]
            upper_neck_link = self.variables["upper_neck_link"]
    
            if lower_neck_link and upper_neck_link:
                kinematic_chain = self.robot_urdf.get_joints_between_links(lower_neck_link, upper_neck_link)
                neck_yaw_joint_dropdown.options = kinematic_chain
                neck_pitch_joint_dropdown.options = kinematic_chain
                if kinematic_chain:
                    neck_yaw_joint_dropdown.value = kinematic_chain[0] if len(kinematic_chain) > 0 else None
                    neck_pitch_joint_dropdown.value = kinematic_chain[1] if len(kinematic_chain) > 1 else None
    
        def on_neck_yaw_joint_change(change):
            self.variables["neck_yaw_joint"] = change.new
    
        def on_neck_pitch_joint_change(change):
            self.variables["neck_pitch_joint"] = change.new
    
        neck_yaw_joint_dropdown.observe(on_neck_yaw_joint_change, names='value')
        neck_pitch_joint_dropdown.observe(on_neck_pitch_joint_change, names='value')
    
        display(
            VBox([
                widgets.HTML("<b>Neck Configuration</b>"),
                lower_neck_output,
                upper_neck_output,
                widgets.HTML("<b>Kinematic Chain Configuration</b>"),
                neck_yaw_output,
                neck_pitch_output
            ])
        )
    
        update_upper_neck_links(torso_end_link)
        update_kinematic_chain()


    def setup_grasp_orientation_gui(self):
        slider_layout = Layout(width="95%")
        button_layout = Layout(width="95%")
        column_layout = Layout(width="50%")
        output_layout = Layout(width="95%", border="1px solid lightgray", padding="10px")
        slider_style = {"description_width": "150px"}
        marker = AxisMarkerPublisher()
        
        def create_arm_control(side):
            tool_frame = self.variables.get(f"{side}_tool_frame")
            arm_identified_var = f"{side}_arm_identified"
            if not self.variables.get(arm_identified_var, False):
                return VBox([widgets.HTML(f"<b>{side.capitalize()} Arm Not Identified</b>")], layout=column_layout)
    
            sliders_output = widgets.Output(layout=output_layout)
            grasp_orientation_output = widgets.Output(layout=output_layout)
            print_output = widgets.Output(layout=output_layout)
    
            def create_slider(joint_name):
                """Create a slider for a joint."""
                joint_limits = self.robot_urdf.joint_limits.get(joint_name, (-math.pi, math.pi))
                slider = widgets.FloatSlider(
                    value=self.robot.get_joint_position(joint_name),
                    min=joint_limits[0],
                    max=joint_limits[1],
                    step=0.01,
                    description=joint_name,
                    layout=slider_layout,
                    style=slider_style
                )
    
                def on_slider_change(change):
                    joint_state = change.new
                    self.robot.set_joint_position(joint_name, joint_state)
    
                slider.observe(on_slider_change, names='value')
                return slider
    
            torso_end_link = self.variables["torso_end_link"]
            arm_end_link = self.variables.get(f"{side}_arm_end_link")
            kinematic_chain = self.robot_urdf.get_joints_between_links(torso_end_link, arm_end_link)
            kinematic_chain = [joint for joint in kinematic_chain if joint in self.robot_urdf.joint_types]
    
            sliders = [create_slider(joint_name) for joint_name in kinematic_chain]
    
            with sliders_output:
                widgets.HTML(f"<b>{side.capitalize()} Arm Control</b>")
                for slider in sliders:
                    display(slider)
    
            def publish_tool_frame(_):
                gripper_tool_frame = self.variables.get(f"{side}_tool_frame")
                with print_output:
                    print_output.clear_output()
                    if gripper_tool_frame:
                        link_pose = self.robot.get_link_pose(gripper_tool_frame)
                        marker.publish([link_pose], length=0.2, duration=10)
                        print(f"{side.capitalize()} Tool Frame published.")
                    else:
                        print(f"No tool frame set for {side.capitalize()} Arm.")
    
            def _set_grasp_orientation(rounded: bool = False):
                with print_output:
                    print_output.clear_output()
                    with simulated_robot:
                        frame = self.variables.get(f"{side}_tool_frame")
                        if not frame:
                            print(f"No tool frame set for {side.capitalize()} Arm.")
                            return
    
                        pose = self.robot.get_link_pose(frame)
                        orientation = pose.orientation_as_list()
                        rounded_orientation = round_quaternion_to_nearest_90(orientation)
                        set_orientation = np.round(rounded_orientation, 3).tolist() if rounded else orientation

                        self.variables[f"{side}_grasp_orientation"] = set_orientation

                        set_alignment = find_aligned_axis(set_orientation)

                        self.variables[f"{side}_palm_axis"] = set_alignment.tolist()
    
                        print(f"{side.capitalize()} Grasp Orientation detected: {orientation}")
                        print(f"{side.capitalize()} Grasp Orientation set to: {set_orientation} (Rounded: {rounded})")
                        print(f"{side.capitalize()} Palm Axis calculated as: {self.variables[f'{side}_palm_axis']}")
                        
            def set_grasp_orientation(_):
                _set_grasp_orientation(rounded=False)
                
            def set_grasp_orientation_rounded(_):
                _set_grasp_orientation(rounded=True)
    
            publish_button = widgets.Button(
                description=f"Publish {side.capitalize()} Tool Frame",
                layout=button_layout
            )
            publish_button.on_click(publish_tool_frame)
    
            set_orientation_button = widgets.Button(
                description=f"Set {side.capitalize()} Grasp Orientation",
                layout=button_layout
            )
            set_orientation_button.on_click(set_grasp_orientation)
            
            set_orientation_rounded_button = widgets.Button(
                description=f"Set {side.capitalize()} Grasp Orientation (Rounded)",
                layout=button_layout
            )
            set_orientation_rounded_button.on_click(set_grasp_orientation_rounded)
    
            return VBox([
                widgets.HTML(f"<b>{side.capitalize()} Gripper Front Grasp Setup</b>"),
                sliders_output,
                VBox([publish_button, set_orientation_button, set_orientation_rounded_button], layout=output_layout),
                print_output
            ], layout=column_layout)
    
        left_arm_control = create_arm_control("left")
        right_arm_control = create_arm_control("right")
    
        display(HBox([left_arm_control, right_arm_control]))


In [3]:
urdf_path = f'../../resources/robots/fetch.urdf'  # Updated URDF path
builder = RobotDescriptionBuilder(urdf_path)

In [4]:
from IPython.display import HTML, display

def apply_global_styles():
    custom_styles = """
    <style>
        /* General Output Styling */
        .jp-OutputArea {
            background: #dff2f8 !important;
            color: #003152 !important;
            border: 1px solid #003152 !important;
        }

        /* Styling text inside output */
        .jp-OutputArea * {
            color: #003152 !important;
        }

        /* Widget Dropdown Styling */
        .widget-dropdown > select {
            background: #dff2f8 !important;
            color: #003152 !important;
            border: 1px solid #003152 !important;
        }

        /* Dropdown Options Styling */
        .widget-dropdown option {
            background: #dff2f8 !important;
            color: #003152 !important;
        }

        /* Label Styling */
        .widget-label {
            color: #003152 !important;
        }

        /* Widget Output */
        .widget-output {
            background: #dff2f8 !important;
            color: #003152 !important;
            border: 1px solid #003152 !important;
        }

        /* Styling for widgets.HTML content */
        .widget-html-content {
            background: #dff2f8 !important;
            color: #003152 !important;
            border: none !important;
            font-weight: bold !important;
        }

        /* Bold text inside widgets */
        .widget-html-content b {
            color: #003152 !important;
        }

        /* Text Widget Styling (widgets.Text, widgets.FloatText) */
        .widget-text input, .widget-float-text input {
            background: #dff2f8 !important;
            color: #003152 !important;
            border: 1px solid #003152 !important;
        }

        /* Button Widget Styling */
        .widget-button {
            background: #dff2f8 !important;
            color: #003152 !important;
            border: 1px solid #003152 !important;
            font-weight: bold !important;
        }

        /* Button Hover State */
        .widget-button:hover {
            background: #cce7f1 !important; /* Slightly darker shade for hover */
        }

        .jp-OutputArea-output {
            max-height: none !important;
            overflow: visible !important;
        }
    </style>
    """
    display(HTML(custom_styles))

# Apply the styles globally
apply_global_styles()


In [5]:
builder.setup_torso_gui()

In [6]:
builder.setup_arm_gui()

In [7]:
builder.setup_arm_park_states()

In [8]:
builder.setup_gripper_configuration()

In [9]:
builder.setup_camera_configuration()

In [10]:
builder.setup_neck_configuration()

In [11]:
builder.setup_grasp_orientation_gui()

In [12]:
builder.debug_variables()

Current Variables State:
robot_name: fetch
root_frame: base_link
torso_start_link: torso_lift_link
torso_start_joint: torso_lift_joint
start_torso_parent_link: base_link
torso_end_link: torso_lift_link
left_arm_end_link: wrist_roll_link
right_arm_end_link: 
left_gripper_parent_link: gripper_link
right_gripper_parent_link: 
left_tool_frame: gripper_link
right_tool_frame: 
left_gripper_open_states: {'r_gripper_finger_joint': 0.05, 'l_gripper_finger_joint': 0.05}
left_gripper_close_states: {'r_gripper_finger_joint': 0.0, 'l_gripper_finger_joint': 0.0}
right_gripper_open_states: {}
right_gripper_close_states: {}
left_arm_park_states: {'wrist_roll_joint': -0.0015926535897929917, 'shoulder_pan_joint': 1.6056, 'shoulder_lift_joint': 1.518, 'upperarm_roll_joint': 3.141592653589793, 'elbow_flex_joint': -1.6909999999999998, 'forearm_roll_joint': -0.0015926535897929917, 'wrist_flex_joint': -1.5100000000000002}
right_arm_park_states: {}
left_gripper_type: GripperType.PARALLEL
right_gripper_type: G

In [13]:
generate_robot_file(variables=builder.variables)

Generated file successfully at: /home/me/IAI_work/master_ws/src/pycram/src/pycram/robot_descriptions/fetch_description.py
